In [42]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import json

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# os.listdir('/kaggle/input/competitions/llm-classification-finetuning')


# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/llm-classification-finetuning/sample_submission.csv
/kaggle/input/competitions/llm-classification-finetuning/train.csv
/kaggle/input/competitions/llm-classification-finetuning/test.csv


In [43]:
# !pip uninstall -y transformers
# !pip install transformers==5.17.0
# import transformers
# print(transformers.__version__)

In [44]:
df = pd.read_csv('/kaggle/input/competitions/llm-classification-finetuning/train.csv')
df.head()

,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [45]:
# print(df.iloc[0].to_dict())

for col in ['prompt', 'response_a', 'response_b']:
    lengths = df[col].astype(str).str.len()
    print(f"{col}: max={lengths.max()}, mean={lengths.mean():.0f}, p50={lengths.quantile(0.5):.0f}")

prompt: max=33056, mean=369, p50=96
response_a: max=54058, mean=1378, p50=1076
response_b: max=53830, mean=1386, p50=1086


In [46]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

checkpoint = "microsoft/deberta-v3-xsmall"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=3, dtype=torch.float32)
print("Loaded OK")

Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-xsmall
Key                                        | Status     | 
-------------------------------------------+------------+-
mask_predictions.classifier.weight         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight    | UNEXPECTED | 
mask_predictions.LayerNorm.weight          | UNEXPECTED | 
mask_predictions.LayerNorm.bias            | UNEXPECTED | 
lm_predictions.lm_head.dense.bias          | UNEXPECTED | 
lm_predictions.lm_head.bias                | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias      | UNEXPECTED | 
mask_predictions.dense.weight              | UNEXPECTED | 
mask_predictions.dense.bias                | UNEXPECTED | 
mask_predictions.classifier.bias           | UNEXPECTED | 
lm_predictions.lm_head.dense.weight        | UNEXPECTED | 
deberta.embeddings.word_embeddings._weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias    

Loaded OK


In [47]:
# num_params = sum(p.numel() for p in model.parameters())
# print(f"{num_params/1e6:.1f}M params")

# sum(p.numel() for p in mdl.parameters() if p.requires_grad)

In [48]:
from datasets import Dataset

dataset = Dataset.from_pandas(df)

In [49]:
# dataset.to_pandas().head()

In [50]:
def to_text(s):
    parts = json.loads(s)
    return " ".join(x if x is not None else "" for x in parts)

def preprocess(example):
    p = tokenizer(to_text(example["prompt"]),     add_special_tokens=False)["input_ids"][:128]
    a = tokenizer(to_text(example["response_a"]), add_special_tokens=False)["input_ids"][:180]
    b = tokenizer(to_text(example["response_b"]), add_special_tokens=False)["input_ids"][:180]
    sep = tokenizer.sep_token_id
    cls = tokenizer.cls_token_id
    ids = [cls] + p + [sep] + a + [sep] + b + [sep]
    return {"input_ids": ids, "attention_mask": [1] * len(ids), "labels": example["label"]}

def add_label(example):
    if example["winner_model_a"] == 1: lbl = 0
    elif example["winner_model_b"] == 1: lbl = 1
    else: lbl = 2
    return {"label": lbl}

In [51]:
dataset = dataset.map(add_label)

Map:   0%|          | 0/57477 [00:00<?, ? examples/s]

In [52]:
# dataset = dataset.select(range(10000))
tokenized = dataset.map(preprocess, remove_columns=dataset.column_names)
split = tokenized.train_test_split(test_size=0.1, seed=42)
tokenized_train_ds = split["train"]
tokenized_val_ds   = split["test"]

Map:   0%|          | 0/57477 [00:00<?, ? examples/s]

In [53]:
from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [54]:
from scipy.special import softmax
from sklearn.metrics import log_loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = softmax(logits, axis=-1)
    ll = log_loss(labels, probs, labels=[0, 1, 2])
    acc = (probs.argmax(axis=-1) == labels).mean()
    return {"log_loss": ll, "accuracy": acc}

In [55]:
training_args = TrainingArguments(
    output_dir="training_dir",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    learning_rate=2e-5,
    num_train_epochs=2,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=64,
    fp16=True,
    logging_steps=50,
    report_to="none",
    metric_for_best_model="log_loss",
    greater_is_better=False,
    load_best_model_at_end=True,
)

In [56]:
trainer = Trainer(model,
                  training_args,
                  train_dataset=tokenized_train_ds,
                  eval_dataset=tokenized_val_ds,
                  data_collator=data_collator,
                  compute_metrics=compute_metrics)

In [57]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Log Loss,Accuracy,Runtime,Samples Per Second,Steps Per Second
1,2.164080,2.153042,1.076521,0.390571,66.462500,86.485000,0.677000
2,2.140509,2.142643,1.071321,0.406228,65.771800,87.393000,0.684000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=3234, training_loss=2.158144441886529, metrics={'train_runtime': 3217.5699, 'train_samples_per_second': 32.154, 'train_steps_per_second': 1.005, 'total_flos': 6515896444315812.0, 'train_loss': 2.158144441886529, 'epoch': 2.0})

In [58]:
model.save_pretrained("/kaggle/working/deberta-xsmall-full")
tokenizer.save_pretrained("/kaggle/working/deberta-xsmall-full")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/kaggle/working/deberta-xsmall-full/tokenizer_config.json',
 '/kaggle/working/deberta-xsmall-full/tokenizer.json')

In [59]:
# import torch
# print("model dtype:", next(model.parameters()).dtype)
# print("embedding dtype:", model.deberta.embeddings.word_embeddings.weight.dtype)
# print("classifier dtype:", model.classifier.weight.dtype)

In [60]:
# print(model.config.torch_dtype)
# print(model.config._name_or_path)

In [61]:
# import shutil, os
# shutil.rmtree("training_dir", ignore_errors=True)
# shutil.rmtree("/kaggle/working/deberta-xsmall-ft", ignore_errors=True)